In [1]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score , precision_score , recall_score,f1_score,classification_report, confusion_matrix
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import matplotlib.pyplot as plt
import seaborn as sns


C:\Users\KIIT0001\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
mlflow.set_tracking_uri("https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow")


In [3]:
import dagshub
dagshub.init(repo_owner='Aayush10671', repo_name='yt-comment-sentiment-analysis', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

Accessing as Aayush10671

Initialized MLflow to track repo "Aayush10671/yt-comment-sentiment-analysis"

Repository Aayush10671/yt-comment-sentiment-analysis initialized!

🏃 View run respected-mink-950 at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/0/runs/b108c0c0ea3a4b918a0e40706c9a27b1
🧪 View experiment at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/0


In [4]:
df = pd.read_csv("preprocessed_data.csv")
df.shape

(36793, 2)

In [5]:
print(df['clean_comment'].isnull().sum())          # count of NaN
print(df['clean_comment'].dtype)                   # should be object or string
print(df['clean_comment'].apply(type).value_counts())  # see if all are strings
print(df['clean_comment'].str.len().value_counts().head()) # check empty strings

131
object
clean_comment
<class 'str'>      36662
<class 'float'>      131
Name: count, dtype: int64
clean_comment
14.0    439
15.0    429
24.0    421
19.0    420
22.0    416
Name: count, dtype: int64


In [6]:
# Drop rows where clean_comment is missing
df = df.dropna(subset=['clean_comment'])
# Remove rows where the comment is empty after stripping
df = df[df['clean_comment'].str.strip() != '']
# Ensure all values are strings (just in case)
df['clean_comment'] = df['clean_comment'].astype(str)

In [7]:
df = df.dropna(subset=['clean_comment'])
df = df[df['clean_comment'].str.strip() != '']
df['clean_comment'] = df['clean_comment'].astype(str)

In [9]:
mlflow.set_experiment("tfidf-trigram maxfeature")

2026/07/25 22:21:22 INFO mlflow.tracking.fluent: Experiment with name 'tfidf-trigram maxfeature' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/a7d1c89313ac4c88bdd5ed018b0bebe7', creation_time=1784998283195, experiment_id='4', last_update_time=1784998283195, lifecycle_stage='active', name='tfidf-trigram maxfeature', tags={}, workspace='default'>

In [11]:
def run_experiment(ngram_range, max_features):

  
    vectorizer = TfidfVectorizer(max_features=max_features,ngram_range=ngram_range)
       

    X = vectorizer.fit_transform(df['clean_comment'])
    y = df['category'].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    with mlflow.start_run():

        
        mlflow.set_tag("model_type", "RandomForestClassifier")

      
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("max_features", max_features)

        model = RandomForestClassifier(
            n_estimators=200,
            max_depth=15,
            random_state=42
        )

        mlflow.log_param("n_estimators", 200)
        mlflow.log_param("max_depth", 15)

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        report = classification_report(
            y_test,
            y_pred,
            output_dict=True
        )

        for label, metrics in report.items():
            if label not in ["accuracy", "macro avg", "weighted avg"]:
                for metric_name, metric_value in metrics.items():
                    mlflow.log_metric(
                        f"{label}_{metric_name}",
                        metric_value
                    )

        cm = confusion_matrix(y_test, y_pred)

        plt.figure(figsize=(8,6))
        sns.heatmap(cm, annot=True, fmt="d")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title("Confusion Matrix")

        plt.savefig("confusion_matrix.png")
        plt.close()

        mlflow.log_artifact("confusion_matrix.png")

        mlflow.sklearn.log_model(
            model,
            name=f"_{max_features}_{ngram_range}"
        )

        print(f"max_features={max_features} | ngram={ngram_range} | Accuracy={accuracy:.4f}")

In [12]:
ngram_ranges = [(1,3)]
max_feature_values = [1000,2000,3000,5000,6000,7000,8000,9000,10000]

for max_features in max_feature_values:
    for ngram_range in ngram_ranges:
        run_experiment(ngram_range, max_features)

2026/07/25 22:25:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


max_features=1000 | ngram=(1, 3) | Accuracy=0.6574
🏃 View run unruly-foal-554 at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/4/runs/30d4d8196ea542fea7893b72099635d1
🧪 View experiment at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/4


2026/07/25 22:26:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


max_features=2000 | ngram=(1, 3) | Accuracy=0.6565
🏃 View run adorable-chimp-492 at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/4/runs/cf8d418efef54186808435f5ad23dd94
🧪 View experiment at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/4


2026/07/25 22:27:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

max_features=8000 | ngram=(1, 3) | Accuracy=0.6508
🏃 View run enchanting-fox-836 at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/4/runs/22cad3a5afba410a8cd06d33581f6296
🧪 View experiment at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/4


2026/07/25 22:33:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


max_features=9000 | ngram=(1, 3) | Accuracy=0.6490
🏃 View run overjoyed-horse-673 at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/4/runs/8fe483b164d940658a540b431b3540d1
🧪 View experiment at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/4


2026/07/25 22:34:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


max_features=10000 | ngram=(1, 3) | Accuracy=0.6524
🏃 View run abrasive-foal-506 at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/4/runs/9011e64275ce4e21854d87bb8c4ff082
🧪 View experiment at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/4
